# Exploring Hetionet

Questions: how big is the graph, which node and edge types does it have, and how are the known treatments (Compound–treats–Disease edges) spread over compounds and diseases? Exploration only: results that matter go into `reports/hetionet-report.qmd`.

In [ ]:
# Run from the project folder, like the scripts (lesson 02).
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

import matplotlib.pyplot as plt
import pandas as pd

nodes = pd.read_csv("data/raw/hetionet/hetionet-v1.0-nodes.tsv", sep="\t")
edges = pd.read_csv("data/raw/hetionet/hetionet-v1.0-edges.sif.gz", sep="\t")
print(f"{len(nodes):,} nodes, {len(edges):,} edges")

In [ ]:
# Nodes per type
nodes["kind"].value_counts()

In [ ]:
# Edges per type (metaedge abbreviation)
edges["metaedge"].value_counts()

In [ ]:
# Known treatments: Compound–treats–Disease (CtD) edges
ctd = edges[edges["metaedge"] == "CtD"]
per_compound = ctd["source"].value_counts()
per_disease = ctd["target"].value_counts()
print(f"{len(ctd)} CtD edges between {per_compound.size} compounds and {per_disease.size} diseases")

# Diseases with the most known treatments
names = nodes.set_index("id")["name"]
per_disease.head(5).rename(index=names)

In [ ]:
#| label: fig-ctd-per-disease
#| fig-cap: "Known treatments (CtD edges) per disease, for the diseases that have at least one."
#| fig-alt: "Histogram of known treatments per disease, from 1 to 68. The bars are spread over 1 to 16 treatments, with a tall bar at 1, and a sparse tail of about ten diseases between 21 and 68."
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(per_disease, bins=range(1, per_disease.max() + 2), color="#eb6834")
ax.set_xlabel("Known treatments per disease")
ax.set_ylabel("Diseases")
plt.show()

Known treatments are spread unevenly: many diseases have only a few, and a handful (hypertension, hematologic cancer, asthma) have dozens. The held-out evaluation has to take this into account, because ranking the diseases that already have many treatments is easy.